# Tag-Anchored Hybrid Mapper — IFRS S1/S2 (test harness)

Maps **requirements** (`evidence_tags`) → **concepts** → **payload data providers**, year-aware, with
precedence (canonical `reporting_kpis` > raw collections), `data_gaps` join, confidence, and full
per-slice provenance (`collection.field = value`).

**Resolution order per requirement:** `report_section` (file partition) → `evidence_tags` bridge (L1, deterministic)
→ lexical/embedding fallback for untagged reqs (L3).

Only the **concept bridge** (Cell 3) is meant for domain-expert editing. Everything else is generic.

In [ ]:
import json, glob, os, re
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter

# ---------------- PATHS (relative to this notebook in notebooks/) ----------------
REQ_DIR     = Path("gen_data/IFRS/ifrs_requirements_kb_outputs_final/section_by_section_requirements/json")
PAYLOAD_DIR = Path("gen_data/payloads_risk")

# payload filename section token -> requirement section_key (read from inside the req file)
SECTION_ALIASES = {"metrics_targets": "metrics_and_targets"}
def canon_section(tok): return SECTION_ALIASES.get(tok, tok)

assert REQ_DIR.exists(),     f"missing {REQ_DIR.resolve()}"
assert PAYLOAD_DIR.exists(), f"missing {PAYLOAD_DIR.resolve()}"
print("req files    :", len(list(REQ_DIR.glob('*.json'))))
print("payload files:", len(list(PAYLOAD_DIR.glob('payload_*.json'))))

In [ ]:
# ================= CONCEPT BRIDGE — the only config domain experts edit =================
# concept -> list of (collection, fields|None, precedence)
#   fields=None  -> use all non-key fields of the record
#   precedence: CANON (reporting_kpis, curated) wins over EVID (raw). metadata corrections rank highest.
CANON, EVID = "canonical", "evidence"

BRIDGE = {
 "ghg_emissions":   [("reporting_kpis",["scope1_2024_tco2e","scope2_market_2024_tco2e","scope2_location_2024_tco2e","scope3_travel_2024_tco2e"],CANON),
                     ("scope1",None,EVID),("scope2",None,EVID),("scope3_travel",None,EVID),
                     ("scope3_categories",["category_number","category_name","emissions_tco2e","included_flag"],EVID),
                     ("ghg_methodology",None,EVID),("scope12_consolidation",None,EVID)],
 "scope_1":         [("reporting_kpis",["scope1_2024_tco2e"],CANON),("scope1",None,EVID),("scope12_consolidation",None,EVID)],
 "scope_2":         [("reporting_kpis",["scope2_market_2024_tco2e","scope2_location_2024_tco2e"],CANON),("scope2",None,EVID)],
 "scope_3":         [("reporting_kpis",["scope3_travel_2024_tco2e"],CANON),("scope3_travel",None,EVID),
                     ("scope3_categories",None,EVID),("financed_emissions_equity",None,EVID),("financed_emissions_sovereign",None,EVID)],
 "financed_emissions":[("reporting_kpis",["financed_emissions_2024_tco2e","carbon_intensity_2024_tco2e_per_meur"],CANON),
                       ("financed_emissions",None,EVID),("financed_emissions_equity",None,EVID),("financed_emissions_sovereign",None,EVID)],
 "targets":         [("reporting_kpis",["target_summary"],CANON),("targets",None,EVID)],
 "carbon_credits":  [("reporting_kpis",["carbon_credit_summary"],CANON),("carbon_credits",None,EVID)],
 "metrics":         [("reporting_kpis",None,CANON),("financial_summary",None,EVID),("scope1",None,EVID),
                     ("scope2",None,EVID),("scope3_categories",None,EVID),("financed_emissions",None,EVID),("internal_carbon_price",None,EVID)],
 "materiality":     [("climate_risk_register",None,EVID),
                     ("value_chain_map",["node_name","materiality_flag","sustainability_theme","climate_exposure_type"],EVID),
                     ("climate_financial_effects",None,EVID),("general_requirements_context",None,EVID)],
 "governance_body": [("reporting_kpis",["governance_maturity"],CANON),("governance",None,EVID),("board_minutes",None,EVID)],
 "management_role": [("governance",None,EVID),("board_minutes",None,EVID)],
 "remuneration":    [("reporting_kpis",["governance_maturity"],CANON),
                     ("governance",["ceo_esg_compensation_pct","ceo_compensation_esg_linked","all_exec_climate_remuneration_pct"],EVID)],
 "risk_process":    [("climate_risk_register",None,EVID)],
 "scenario_analysis":[("climate_scenarios",None,EVID),("resilience_assessment",None,EVID)],
 "financial_effects":[("climate_financial_effects",None,EVID),("financial_summary",None,EVID)],
 "business_model_value_chain":[("value_chain_map",None,EVID),("transition_plan",None,EVID),("climate_opportunities",None,EVID)],
 "strategy_decision_making":[("transition_plan",None,EVID),("resilience_assessment",None,EVID),("climate_opportunities",None,EVID)],
 "connected_information":[("financial_summary",None,EVID),("climate_financial_effects",None,EVID),("general_requirements_context",None,EVID)],
 "commercial_banking":[("financed_emissions",None,EVID),("financial_summary",None,EVID)],
 "asset_management":[("financed_emissions_equity",None,EVID)],
 "source_guidance": [("ghg_methodology",None,EVID),("general_requirements_context",None,EVID)],
 # "insurance": intentionally absent -> coverage report flags a missing bridge entry (BANK01 has no insurance data)
}
PREC = {"metadata_correction": -1, CANON: 0, EVID: 1}  # lower wins

def unit_of(f):
    for suf,u in [("_tco2e","tCO2e"),("_meur","MEUR"),("_pct","%"),("_eur","EUR"),("_c","degC")]:
        if f.endswith(suf): return u
    return None
def id_field_of(rec):
    for k in rec:
        if k.endswith("_id") and k!="bank_id": return k
    return "bank_id"

# Optional: externalise the bridge to YAML (config-as-code, PR-reviewable)
try:
    import yaml
    Path("concept_bridge.yaml").write_text(yaml.safe_dump(
        {c:[{"collection":a,"fields":b,"precedence":p} for a,b,p in v] for c,v in BRIDGE.items()},
        sort_keys=False, allow_unicode=True), encoding="utf-8")
    print("wrote concept_bridge.yaml")
except Exception as e:
    print("yaml export skipped:", e)
print("concepts in bridge:", len(BRIDGE))

In [ ]:
def load_requirements(section_key):
    for f in REQ_DIR.glob("*.json"):
        try: d = json.loads(f.read_text(encoding="utf-8"))
        except Exception: continue
        if d.get("section_key") == section_key:
            reqs = []
            for std, blob in d.get("standards", {}).items():
                reqs += blob.get("requirements", [])
            return reqs
    return []

def load_payloads():
    P = {}
    for f in PAYLOAD_DIR.glob("payload_*.json"):
        name = f.stem[len("payload_"):]          # BANK01_metrics_targets
        bank, tok = name.split("_", 1)
        P.setdefault(bank, {})[canon_section(tok)] = json.loads(f.read_text(encoding="utf-8"))
    return P

PAYLOADS = load_payloads()
BANKS = sorted(PAYLOADS)
print("banks:", BANKS)
for b in BANKS: print(" ", b, "->", sorted(PAYLOADS[b]))

In [ ]:
# L3 fallback for UNTAGGED requirements. Lexical (zero-dep) by default.
# To use real embeddings, set USE_EMBEDDINGS=True and plug your encoder (BGE-M3 / MPNet) below.
USE_EMBEDDINGS = False
CONCEPT_LEX = {c: set(re.findall(r"[a-z0-9]+", c)) for c in BRIDGE}

def infer_concepts_lexical(text):
    toks = set(re.findall(r"[a-z0-9]+", (text or "").lower()))
    scored = []
    for c, lex in CONCEPT_LEX.items():
        ov = len(lex & toks)
        if ov: scored.append((ov / max(len(lex), 1), c))
    scored.sort(reverse=True)
    return [(c, min(0.6, s)) for s, c in scored[:2]]

def infer_concepts_embedding(text):
    # HOOK: encode `text` and each concept description, return [(concept, cosine), ...]
    # from sentence_transformers import SentenceTransformer, util  # BGE-M3 / MPNet
    raise NotImplementedError("wire your encoder here")

def infer_concepts(text):
    return infer_concepts_embedding(text) if USE_EMBEDDINGS else infer_concepts_lexical(text)

In [ ]:
def select_records(payload, collection, year, include_comparatives=False):
    if collection not in payload: return []
    v = payload[collection]
    if isinstance(v, dict): return [v]                 # reporting_kpis / *_context
    if not isinstance(v, list): return []
    out = []
    for r in v:
        if not isinstance(r, dict): continue
        if "reporting_year" in r and not include_comparatives and r["reporting_year"] != year:
            continue                                    # year-aware: drop wrong-year rows
        out.append(r)
    return out

def match_gaps(concepts, data_gaps, year):
    prov_fields, prov_cols = set(), set()
    for c in concepts:
        for coll, fields, _ in BRIDGE.get(c, []):
            prov_cols.add(coll)
            if fields: prov_fields |= set(fields)
    hits = []
    for g in data_gaps or []:
        fld = g.get("field", ""); base = fld.split(".")[0]
        if fld in prov_fields or base in prov_cols or base in prov_fields or any(base in f for f in prov_fields):
            if (not g.get("affected_years")) or (year in g["affected_years"]):
                hits.append(g)
    return hits

In [ ]:
@dataclass
class Evidence:
    requirement_id: str; concept: str; collection: str; record_id: str
    field: str; value: object; unit: str; reporting_year: object
    precedence: int; layer: str; is_primary: bool = False

def resolve(req, payload, year):
    tags = req.get("evidence_tags") or []
    if tags:
        layer, concepts = "L1", [(t, 1.0) for t in tags]
    else:
        layer, concepts = "L3_lexical", infer_concepts(req.get("requirement_text", ""))

    slices, concept_conf = [], {}
    for c, base in concepts:
        concept_conf[c] = base
        for coll, fields, prec_kind in BRIDGE.get(c, []):
            for rec in select_records(payload, coll, year):
                use = fields if fields else [k for k in rec if k != "bank_id"]
                rid = rec.get(id_field_of(rec), rec.get("bank_id"))
                ry  = rec.get("reporting_year", year)
                for f in use:
                    val = rec.get(f)
                    if val is None or (isinstance(val, (dict, list)) and not val): continue
                    slices.append(Evidence(req["requirement_id"], c, coll, str(rid), f, val,
                                           unit_of(f), ry, PREC[prec_kind], layer))
    # primary = min-precedence slice(s) per concept (canonical wins; raw kept for traceability)
    byc = defaultdict(list)
    for s in slices: byc[s.concept].append(s)
    for c, ss in byc.items():
        m = min(x.precedence for x in ss)
        for x in ss:
            if x.precedence == m: x.is_primary = True

    gaps        = match_gaps([c for c,_ in concepts], payload.get("metadata", {}).get("data_gaps", []), year)
    no_provider = [c for c,_ in concepts if c not in BRIDGE]
    if   slices: status = "resolved"
    elif gaps:   status = "declared_gap"
    elif no_provider and len(no_provider) == len(concepts): status = "no_bridge_entry"
    else:        status = "unmapped"

    conf = round(max([concept_conf[c] for c in byc] or [0.0]) * req.get("requirement_quality_score", 1.0), 3)
    return dict(requirement_id=req["requirement_id"], standard=req.get("standard"),
                mandatory=req.get("mandatory"), banking_relevance=req.get("banking_relevance"),
                tags=tags, layer=layer, status=status, confidence=conf,
                n_evidence=len(slices), n_primary=sum(s.is_primary for s in slices),
                concepts_no_provider=no_provider, gap_fields=[g.get("field") for g in gaps],
                evidence=[asdict(s) for s in slices])

def run_bank_section(bank, section_key):
    payload = PAYLOADS[bank][section_key]
    year = payload.get("metadata", {}).get("reporting_year")
    reqs = load_requirements(section_key)
    return [resolve(r, payload, year) for r in reqs], year

In [ ]:
BANK = BANKS[0]   # change to loop over BANKS for the full corpus
all_maps = []
print(f"=== {BANK} ===")
for sec in sorted(PAYLOADS[BANK]):
    maps, year = run_bank_section(BANK, sec)
    all_maps += maps
    c = Counter(m["status"] for m in maps)
    print(f"{sec:22s} y={year} reqs={len(maps):3d}  " + "  ".join(f"{k}={v}" for k,v in c.most_common()))

print("\nOVERALL:", dict(Counter(m['status'] for m in all_maps)), "TOTAL", len(all_maps))
res = [m for m in all_maps if m['status']=='resolved']
print("resolved:", len(res), "| avg evidence slices:", round(sum(m['n_evidence'] for m in res)/max(len(res),1),1))

In [ ]:
# ---------- COVERAGE: unmapped requirements / missing bridge entries / unused data ----------
REL = {"high":0,"medium":1,"low":2}
unmapped = [m for m in all_maps if m['status'] in ('unmapped','no_bridge_entry')]
unmapped.sort(key=lambda m:(not m['mandatory'], REL.get(m['banking_relevance'],3)))
print(f"UNMAPPED / NO-BRIDGE: {len(unmapped)}  (mandatory: {sum(m['mandatory'] for m in unmapped)})")
for m in unmapped[:15]:
    print(f"  {m['requirement_id']:16s} mand={str(m['mandatory']):5s} rel={m['banking_relevance']:6s} "
          f"tags={m['tags']} status={m['status']}")

miss = Counter()
for m in all_maps:
    for c in m['concepts_no_provider']: miss[c]+=1
print("\nMISSING BRIDGE ENTRIES (tag seen, no provider):", dict(miss))

# unused data: collections present in a section payload but never emitted as evidence for it
print("\nUNUSED COLLECTIONS per section (candidate dead data or missing tag):")
for sec in sorted(PAYLOADS[BANK]):
    maps,_ = run_bank_section(BANK, sec)
    used = {e['collection'] for m in maps for e in m['evidence']}
    present = {k for k,v in PAYLOADS[BANK][sec].items()
               if k not in ('bank','metadata') and (isinstance(v,(list,dict)) and v)}
    unused = sorted(present - used)
    if unused: print(f"  {sec:22s}: {unused}")

In [ ]:
# ---------- POST-MAP VALIDATION ----------
def v_mandatory_coverage(maps):
    bad = [m['requirement_id'] for m in maps if m['mandatory'] and m['status'] in ('unmapped','no_bridge_entry')]
    return ("mandatory_coverage", not bad, f"{len(bad)} mandatory reqs without evidence or declared gap", bad[:10])

def v_year_integrity(maps, year):
    bad = [(m['requirement_id'], e['collection'], e['reporting_year'])
           for m in maps for e in m['evidence']
           if isinstance(e['reporting_year'], int) and e['reporting_year'] != year]
    return ("year_integrity", not bad, f"{len(bad)} evidence slices from a non-reporting year", bad[:5])

def v_scope1_arithmetic(payload, year):
    fails=[]
    for r in payload.get("scope1", []):
        if r.get("reporting_year")==year:
            if abs(r.get("scope1_gas_tco2e",0)+r.get("scope1_fleet_tco2e",0)-r.get("scope1_total_tco2e",0))>1e-4:
                fails.append(r)
    return ("scope1_gas+fleet==total", not fails, f"{len(fails)} rows fail", fails)

def v_attribution_bounds(payload):
    bad=[i.get("investment_id") for i in payload.get("financed_emissions_equity",[])+payload.get("financed_emissions_sovereign",[])
         if not (0 <= i.get("attribution_factor",0) <= 1)]
    return ("attribution_factor_in_[0,1]", not bad, f"{len(bad)} out of bounds", bad)

def v_canonical_vs_raw(payload, year):
    """conflict detector: canonical reporting_kpis vs raw collection for the same quantity"""
    out=[]
    kpi=payload.get("reporting_kpis",{})
    raw=[r for r in payload.get("scope1",[]) if r.get("reporting_year")==year]
    if raw and "scope1_2024_tco2e" in kpi:
        d=abs(kpi["scope1_2024_tco2e"]-raw[0].get("scope1_total_tco2e",0))
        out.append(("scope1 kpi vs raw", d<1e-3, f"|delta|={d:.4f} (precedence: canonical wins, raw kept as evidence)"))
    return out

for sec in sorted(PAYLOADS[BANK]):
    maps, year = run_bank_section(BANK, sec)
    pl = PAYLOADS[BANK][sec]
    checks = [v_mandatory_coverage(maps), v_year_integrity(maps, year)]
    if "scope1" in pl: checks += [v_scope1_arithmetic(pl, year)]
    if "financed_emissions_equity" in pl or "financed_emissions_sovereign" in pl: checks += [v_attribution_bounds(pl)]
    conflicts = v_canonical_vs_raw(pl, year) if "scope1" in pl else []
    print(f"--- {sec} ---")
    for name, ok, msg, *rest in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}: {msg}")
    for name, ok, msg in conflicts:
        print(f"  [{'ok' if ok else 'CONFLICT'}] {name}: {msg}")

In [ ]:
# ---------- EVIDENCE TRACE: full provenance for one requirement ----------
def trace(requirement_id):
    m = next((x for x in all_maps if x['requirement_id']==requirement_id), None)
    if not m: print("not found"); return
    print(f"{m['requirement_id']}  status={m['status']}  conf={m['confidence']}  layer={m['layer']}")
    print(f"  tags={m['tags']}  mandatory={m['mandatory']}  gaps={m['gap_fields']}")
    for e in m['evidence']:
        star = "*" if e['is_primary'] else " "
        u = f" [{e['unit']}]" if e['unit'] else ""
        print(f"  {star} {e['concept']:22s} {e['collection']}.{e['field']} = {e['value']}{u}"
              f"  (rec={e['record_id']}, y={e['reporting_year']}, prec={e['precedence']})")

# example: first resolved requirement
trace(next(m['requirement_id'] for m in all_maps if m['status']=='resolved'))

### Export — reference mode (lean, UTF-8, fully traceable)

On disk we persist **references** (`requirement -> concept -> collection -> record_ids`) plus one
`evidence_store` per bank holding each record **once**. Round-trip: mapping -> store -> values.
This is ~40x smaller than inlining every field value into every requirement, and portable
(explicit UTF-8, so `§B62` etc. survive on Windows). The in-memory `all_maps` above stays
field-level for interactive coverage/validation/trace.

In [ ]:
# ---------- EXPORT: reference mode ----------
NOISE = {"bank_id","reporting_year","is_synthetic","data_source","currency"}
def disclosure_fields(rec):
    idf = id_field_of(rec)
    return [k for k,v in rec.items()
            if k not in NOISE and k != idf and v is not None
            and not (isinstance(v,(dict,list)) and not v)]

def resolve_refs(req, payload, year, store):
    tags = req.get("evidence_tags") or []
    if tags: layer, concepts = "L1", [(t,1.0) for t in tags]
    else:    layer, concepts = "L3_lexical", infer_concepts(req.get("requirement_text",""))
    agg = defaultdict(lambda: {"record_ids": [], "precedence": 9})
    for c,_ in concepts:
        for coll, fields, prec_kind in BRIDGE.get(c, []):
            for rec in select_records(payload, coll, year):
                if not (fields or disclosure_fields(rec)): continue
                rid = str(rec.get(id_field_of(rec), rec.get("bank_id")))
                store[coll][rid] = rec                        # record stored ONCE
                a = agg[(c, coll)]
                if rid not in a["record_ids"]: a["record_ids"].append(rid)
                a["precedence"] = min(a["precedence"], PREC[prec_kind])
    refs = [dict(concept=c, collection=coll, precedence=v["precedence"], record_ids=v["record_ids"])
            for (c,coll), v in agg.items()]
    for c in {r["concept"] for r in refs}:
        cs = [r for r in refs if r["concept"]==c]; mn = min(r["precedence"] for r in cs)
        for r in cs: r["is_primary"] = (r["precedence"]==mn)
    gaps = match_gaps([c for c,_ in concepts], payload.get("metadata",{}).get("data_gaps",[]), year)
    nop  = [c for c,_ in concepts if c not in BRIDGE]
    status = "resolved" if refs else ("declared_gap" if gaps else
             ("no_bridge_entry" if nop and len(nop)==len(concepts) else "unmapped"))
    conf = round((1.0 if tags else max([s for _,s in concepts] or [0.0])) * req.get("requirement_quality_score",1.0), 3)
    return dict(requirement_id=req["requirement_id"], standard=req.get("standard"),
                mandatory=req.get("mandatory"), banking_relevance=req.get("banking_relevance"),
                tags=tags, layer=layer, status=status, confidence=conf,
                n_records=sum(len(r["record_ids"]) for r in refs),
                gap_fields=[g.get("field") for g in gaps], concepts_no_provider=nop,
                evidence_refs=refs)

OUT = Path("mapping_outputs"); OUT.mkdir(exist_ok=True)
for bank in BANKS:
    store = defaultdict(dict); payload_maps = []
    for sec in sorted(PAYLOADS[bank]):
        payload = PAYLOADS[bank][sec]; year = payload.get("metadata",{}).get("reporting_year")
        for r in load_requirements(sec):
            payload_maps.append({"section": sec, **resolve_refs(r, payload, year, store)})
    (OUT / f"mapping_{bank}.json").write_text(
        json.dumps(payload_maps, ensure_ascii=False, indent=2), encoding="utf-8")          # <- UTF-8
    (OUT / f"evidence_store_{bank}.json").write_text(
        json.dumps({k: dict(v) for k,v in store.items()}, ensure_ascii=False, indent=2), encoding="utf-8")
    mb = (OUT/f"mapping_{bank}.json").stat().st_size/1e6
    sb = (OUT/f"evidence_store_{bank}.json").stat().st_size/1e6
    print(f"{bank}: mapping {mb:.2f} MB + store {sb:.2f} MB  ({len(payload_maps)} reqs)")

def trace_disk(bank, requirement_id):
    maps  = json.loads((OUT/f"mapping_{bank}.json").read_text(encoding="utf-8"))
    store = json.loads((OUT/f"evidence_store_{bank}.json").read_text(encoding="utf-8"))
    m = next((x for x in maps if x["requirement_id"]==requirement_id), None)
    if not m: print("not found"); return
    print(f"{requirement_id} status={m['status']} conf={m['confidence']} tags={m['tags']}")
    for ref in m["evidence_refs"]:
        star = "*" if ref.get("is_primary") else " "
        for rid in ref["record_ids"][:3]:
            rec = store[ref["collection"]][rid]
            shown = {k: rec[k] for k in disclosure_fields(rec)[:4]}
            print(f"  {star} {ref['concept']:20s} {ref['collection']}[{rid}] -> {shown}")

trace_disk(BANKS[0], next(m["requirement_id"] for m in all_maps if m["status"]=="resolved"))